In [ ]:
import pandas as pd
import numpy as np
import os
import glob as glob
import sys
import json

parent_dir = os.path.abspath(os.path.join(os.path.dirname(os.getcwd())))
sys.path.append(parent_dir)
import cpg_harmonizer
import s3_loader
import harmonize_checker

In [ ]:
%load_ext autoreload
%autoreload 2

# Enter Identifiers, Per-Dataset Metadata

In [ ]:
project_name = "" # e.g. cpg0016-jump
source_list = [''] # e.g. ['broad'] or ['source_1','source_2']
output_parent_directory = "/path/to/output/parent/folder"

# typically, per-dataset metadata
# if not dataset-wide, delete from here and create conditional entry below
# if unknown, comment out
per_dataset_manual = {
    'Plate_Size':000,
    'CP_Version':'', # e.g. "v1"
    'DOI_to_Cite':'12.3456/1234-5678.example-DOI',
    'Year_Imaged': 0000,
    'Cell_Line_Name':'', # e.g. "HeLa"
    'Cell_Line_Type':'',
    'Cell_Line_Modification':'', # e.g. "None"
    'Microscope_Name':'',
    'Microscope_Binning': 0,
    'Microscope_Modality':'', # e.g. 'Widefield
    'Microscope_Objective_Magnification':0, # e.g. 20
    'Microscope_Objective_NA':0, # e.g. .45
    'Microscope_Pixel_Size': 0,
    'Image_Bit_Depth': 0, # e.g. 16
    'Image_Size_X':0000,
    'Image_Size_Y':0000,
    'Timepoint_Primary_Treatment':0,
    'Timepoint_Secondary_Treatment':0,
    'Timepoint_Acquisition':0,
    'Treatment_Category':'', #e.g. 'Compound'
}

In [ ]:
# excitation/emission values used for acquisition
# {label:{'ex_peak':value,'ex_width':value,'em_peak':value,'em_width':value}}
# if value is unknown, define value with np.nan
# can delete this cell if all values are unknown
ex_em_dict = {
    'DNA':{'ex_peak':387,
           'ex_width':np.nan,
           'em_peak':447,
           'em_width':np.nan},
    'ER':{'ex_peak':472,
          'ex_width':np.nan,
           'em_peak':520,
           'em_width':np.nan},
    'RNA':{'ex_peak':531,
           'ex_width':np.nan,
           'em_peak':593,
           'em_width':np.nan},
    'AGP':{'ex_peak':562,
           'ex_width':np.nan,
           'em_peak':642,
           'em_width':np.nan},
    'Mito':{'ex_peak':628,
            'ex_width':np.nan,
           'em_peak':692,
           'em_width':np.nan}
}

# Join CPG Metadata

In [ ]:
if len(source_list) == 1:
    output_directory = os.path.join(output_parent_directory, project_name, source_list[0], "workspace", "harmonized_metadata")
else:
    output_directory = os.path.join(output_parent_directory, project_name, "all", "workspace", "harmonized_metadata")
if not os.path.exists(output_directory):
    os.makedirs(output_directory, exist_ok=True)

metadata_paths = []
load_paths = []
for src in source_list:
    metadata_paths.extend(s3_loader.parse_s3_folder(f"{project_name}/{src}/workspace/metadata/platemaps/"))
    load_paths.extend(s3_loader.parse_s3_folder(f"{project_name}/{src}/workspace/load_data_csv/"))

project = cpg_harmonizer.Project(output_directory, "../output_structure.json", project_name=project_name)

In [ ]:
main_csvs = []
main_csv_batch = []
for path in load_paths:
    if path.endswith("load_data.csv"):
        main_csvs.append(s3_loader.read_s3_file(path, sep = ","))
        main_csv_batch.append(path.split("/")[-3])

In [ ]:
platemap_txt = []
platemap_csvs = []
platemap_csv_batch = []
platemap_txt_batch = []
platemap_txt_name = []
external_tsv = []


for path in metadata_paths:
    if path.endswith(".csv"):
        platemap_csvs.append(s3_loader.read_s3_file(path, sep = ","))
        platemap_csv_batch.append(path.split("/")[-2])
    if path.endswith(".txt"):
        platemap_txt.append(s3_loader.read_s3_file(path, sep = "\t"))
        platemap_txt_name.append(path.split("/")[-1].split(".")[0])
        platemap_txt_batch.append(path.split("/")[-3])
    if path.endswith(".tsv"):
        external_tsv.append(s3_loader.read_s3_file(path, sep = "\t"))

In [ ]:
complete_df = project.run_conversion(
    main_csvs = main_csvs, 
    main_csv_batch = main_csv_batch, 
    platemap_csvs = platemap_csvs,  
    platemap_csv_batch = platemap_csv_batch, 
    platemap_txt= platemap_txt, 
    platemap_txt_batch = platemap_txt_batch,
    platemap_txt_name = platemap_txt_name,
    external_tsv = external_tsv
)

# Additional cleaning steps

In [ ]:
with open('../inferable_relationships.json', "r") as f:
    inferable_metadata = json.load(f)

if per_dataset_manual['CP_Version'] != 'other':
    if not all([x in inferable_metadata["Label"].keys() for x in complete_df['Label'].unique()]):
        print(f'Labels need to be corrected to match any of {list(inferable_metadata["Label"].keys())}')
        print(f"Current labels are {complete_df['Label'].unique()}")
    # infer metadata from known relationships
    for column in inferable_metadata:
        for entry in inferable_metadata[column]:
            for inferred_column in inferable_metadata[column][entry]:
                if not inferred_column in complete_df.columns:
                    complete_df[inferred_column] = np.nan
                    complete_df[inferred_column] = complete_df[inferred_column].astype('str')
                complete_df.loc[complete_df[column]==entry, inferred_column] = inferable_metadata[column][entry][inferred_column]
else:
    print("Did not infer label metadata because CP_Version not inferrable")
    print(f"Labels that need to be manually declared are {complete_df['Label'].unique()}")

In [ ]:
# add per label excitation and emission values
# can delete if all values are unknown
if set(complete_df['Label'].unique()) == ex_em_dict.keys():
    for key in ex_em_dict:
        complete_df.loc[complete_df["Label"] == key, "Microscope_Excitation_Peak"] = ex_em_dict[key]['ex_peak']
        complete_df.loc[complete_df["Label"] == key, "Microscope_Excitation_Width"] = ex_em_dict[key]['ex_width']
        complete_df.loc[complete_df["Label"] == key, "Microscope_Emission_Peak"] = ex_em_dict[key]['em_peak']
        complete_df.loc[complete_df["Label"] == key, "Microscope_Emission_Width"] = ex_em_dict[key]['em_width']

else:
    print("Excitation/Emission dictionary does not match Labels")
    print(f"Available labels are {set(complete_df['Label'].unique())}")
    print(f"Defined keys are {ex_em_dict.keys()}")

In [ ]:
# add per-experiment metadata manually annotated above
for col, val in per_dataset_manual.items():
    complete_df[col] = val

# add source information inferred from file path
for source in source_list:
    complete_df.loc[complete_df['File Path'].str.contains(f"/{source}/"),'Source'] = source

In [ ]:
# manually declare labeling information for non-canonical protocol
"""
# example conditional label information
complete_df.loc[complete_df["Label"] == "Tubulin", "Labeling_Reagent"] = "mouse anti–β-tubulin IgG and Alexa Fluor 488 donkey anti-mouse IgG"
complete_df.loc[complete_df["Label"] == "Tubulin", "Labeled_Molecule"] = "β-tubulin"
complete_df.loc[complete_df["Label"] == "Tubulin", "Labeled_Structure"] = "β-tubulin"
complete_df.loc[complete_df["Label"] == "Tubulin", "Label_Mechanism"] = "Antibody"
"""

In [ ]:
# report on un-harmonized columns
# use reported information to manually update ontology OR dataframe OR input metadata
# this cell does NOT save the harmonization results, just check them
# this allows you to make any necessary corrections without accidentally overwriting
# returns a view of what the dataframe will look like after final harmonization
extra_cols = harmonize_checker.check_columns('../harmonized_ontology.json',complete_df, ret="extra_cols")

print("View of data that will be kept")
complete_df[[x for x in complete_df.columns if x not in extra_cols]]

In [ ]:
print("View of data that will be removed")
complete_df[extra_cols]

In [ ]:
# After correcting any warnings above, run to save harmonization
complete_df = harmonize_checker.check_columns('../harmonized_ontology.json',complete_df)

In [ ]:
saved_path = os.path.join(output_directory,f"{project_name}_harmonized_metadata.parquet")
complete_df.to_parquet(saved_path)